In [ ]:
import numpy as np
import anndata as an
import scanpy as sc
import scanpy.external as sce
import scipy
import os
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn.functional as F
import pandas as pd
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests
import PyComplexHeatmap as pch
import pandas as pd

In [ ]:
qs = torch.load('results/heart/qs.pt',map_location='cpu',weights_only = False)
h  = torch.load('results/heart/h.pt',map_location='cpu',weights_only = False)
z  = torch.load('results/heart/zs.pt',map_location='cpu',weights_only = False)
Cs  = torch.load('results/heart/Cs.pt',map_location='cpu',weights_only = False)

In [ ]:
chaffin, kuppe, koenig, reichart, simonson = [np.argmax(q, axis=1) for q in qs]

In [ ]:
chaffin_label, kuppe_label, koenig_label, reichart_label, simonson_label = [
    [str(x) for x in np.argmax(q.detach().numpy(), axis=1)] for q in qs
]
qs = [x.detach().numpy() for x in qs]

In [ ]:
chaffin, kuppe, koenig, reichart, simonson = [
    x.detach().numpy() for x in [chaffin, kuppe, koenig, reichart, simonson]
]

In [ ]:
pairs = study_pairs = [(0,1), (1,2), (2,3), (3,4), (4,0), (0,2), (0,3), (1,3), (1,4), (2,4)]
def normalize(A, B):
    C = (A + B.T) / 2
    return (C - C.min()) / (C.max() - C.min())
C = [normalize(Cs[i][j], Cs[j][i]) for i,j in study_pairs]

In [ ]:
adata = an.AnnData(h[0].detach().numpy())
adata.obs['confidence'] = qs[0][np.arange(len(qs[0])), qs[0].argmax(1)]
adata.obs['cluster'] = chaffin_label
sc.pp.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata)
sc.pl.umap(adata,color=['cluster','confidence'],color_map='viridis',title='UMAP of Chaffin Clusters in Latent Space')

In [ ]:
adata = an.AnnData(h[1].detach().numpy())
adata.obs['confidence'] = qs[1][np.arange(len(qs[1])), qs[1].argmax(1)]
adata.obs['cluster'] = kuppe_label
sc.pp.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata)
sc.pl.umap(adata,color=['cluster','confidence'],color_map='viridis',title='UMAP of Kuppe Clusters in Latent Space')

In [ ]:
adata = an.AnnData(h[2].detach().numpy())
adata.obs['confidence'] = qs[2][np.arange(len(qs[2])), qs[2].argmax(1)]
adata.obs['cluster'] = koenig_label
sc.pp.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata)
sc.pl.umap(adata,color=['cluster','confidence'],color_map='viridis',title='UMAP of Kuppe Clusters in Latent Space')

In [ ]:
adata = an.AnnData(h[3].detach().numpy())
adata.obs['cluster'] = reichart_label
adata.obs['confidence'] = qs[3][np.arange(len(qs[3])), qs[3].argmax(1)]
sc.pp.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata)
sc.pl.umap(adata,color=['cluster','confidence'],color_map='viridis',title='UMAP of Reichart Clusters in Latent Space')

In [ ]:
adata = an.AnnData(h[4].detach().numpy())
adata.obs['confidence'] = qs[4][np.arange(len(qs[4])), qs[4].argmax(1)]
adata.obs['cluster'] =simonson_label
sc.pp.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata)
sc.pl.umap(adata,color=['cluster','confidence'],color_map='viridis',title='UMAP of Simonson Clusters in Latent Space')

In [ ]:
def fishers_test(organ1,organ2):
    fisher_lk = []
    ps = []
    for i in np.unique(organ1):
        for j in np.unique(organ2):
            a = np.sum((organ1 == i) & (organ2 ==j))
            b = np.sum((organ1 == i) & (organ2 !=j))
            c = np.sum((organ1 != i) & (organ2 ==j))
            d = np.sum((organ1 != i) & (organ2 !=j))
            table = [[a, b],
                     [c, d]]
            _, p = fisher_exact(table,alternative='greater')
            ps.append(p)
    ps = np.array(ps)
    _, ps, _, _ = multipletests(ps, alpha=0.05, method='fdr_bh')
    ps = ps.reshape(len(np.unique(organ1)),len(np.unique(organ2)))
    return ps
def star(p):
    if p <= 0.01:
        return '**'
    elif p <= 0.05:
        return '*'
    else:
        return ''
        
def plot_C(study1, study2, study1_name, study2_name, C):
    ps = fishers_test(study1,study2)
    data = pd.DataFrame(C)
    ps = pd.DataFrame(ps)
    ps = ps.map(star)

    plt.figure(figsize=(6, 4))
    sns.heatmap(
       data,
        cmap='Blues',
        vmin=0, vmax=1,
        annot=ps,
        annot_kws={'fontsize': 10, 'color': 'red', 'weight': 'bold'},
        fmt='s',
        xticklabels=np.unique(study2), 
        yticklabels=np.unique(study1), 
    )
    plt.title(f"Association between {study1_name} and {study2_name} Clusters", fontsize=10)
    plt.text(8, 1, '* : p <= 0.05', fontsize=10)
    plt.text(8, 1.5, '**: p <= 0.01', fontsize=10)
    plt.xlabel(study2_name)
    plt.ylabel(study1_name)  
    plt.tight_layout()
    plt.show()

In [ ]:
pairs = [
    (chaffin, kuppe, 'chaffin', 'kuppe'),
    (kuppe, koenig, 'kuppe', 'koenig'),
    (koenig, reichart, 'koenig', 'reichart'),
    (reichart, simonson, 'reichart', 'simonson'),
    (simonson, chaffin, 'simonson', 'chaffin'),
    (chaffin, koenig, 'chaffin', 'koenig'),
    (chaffin, reichart, 'chaffin', 'reichart'),
    (kuppe, reichart, 'kuppe', 'reichart'),
    (kuppe, simonson, 'kuppe', 'simonson'),
    (koenig, simonson, 'koenig', 'simonson')
]

for i, (s1, s2, name1, name2) in enumerate(pairs):
    plot_C(s1, s2, name1, name2, C[i])

In [ ]:
pairs = [(0,1), (1,2), (2,3), (3,4), (4,0), (0,2), (0,3), (1,3), (1,4), (2,4)]
mat = pd.DataFrame(np.zeros((30, 30)))
i = 0
for (x,y) in pairs:
    
    row = x*6
    col = y*6
    mat.iloc[row:row+6, col:col+6] = np.array(C[i])
    i = i+1

for i in range(5): 
    row = i * 6
    col = i * 6
    mat.iloc[row:row+6, col:col+6] = np.identity(6)


In [ ]:
mat = mat.to_numpy()
mat = np.maximum(mat, mat.T)
mat

In [ ]:
studies = ['chaffin', 'kuppe', 'koenig', 'reichart', 'simonson']
#cell_types =['lymphoid','myeloid','mesenchymal','endothelial','myeloid','mesenchymal']

cell_types =np.array([['lymphoid','myeloid','mesenchymal','endothelial','myeloid','mesenchymal'],
             ['endothelial','mesenchymal','mesenchymal','lymphoid','myeloid','myeloid'],
             ['mesenchymal','mesenchymal','lymphoid','myeloid','endothelial','myeloid'],
             ['endothelial','mesenchymal','myeloid','myeloid','mesenchymal','lymphoid'],
             ['lymphoid','mesenchymal','mesenchymal','myeloid','myeloid','endothelial']])
s_rows = [s for s in studies for k in range(6)]  # study_cols the same
#clusters = np.array([[0,1,2,3,4,5],[3,5,2,0,4,1],[2,3,1,4,5,0],[5,2,4,0,3,1],[0,4,2,5,3,1]])
clusters = np.array([[0,1,2,3,4,5],[0,1,2,3,4,5],[0,1,2,3,4,5],[0,1,2,3,4,5],[0,1,2,3,4,5]])

df = pd.DataFrame(s_rows,columns = ['studies'])
df['cell_types'] = cell_types.flatten()
df['cluster'] = clusters.astype(str).flatten()

In [ ]:
col_ha = pch.HeatmapAnnotation(
    #label = pch.anno_label(df.cluster, merge=False,rotation=15),
    #cluster = pch.anno_simple(df.cluster,add_text=False,legend=True),
    studies=pch.anno_simple(df.studies,add_text=False,legend=True),
    cell_types=pch.anno_simple(df.cell_types, add_text=False,legend=True),axis=1,                            
)
row_ha = pch.HeatmapAnnotation(
    #label = pch.anno_label(df.cluster, merge=False,rotation=15),
    #cluster = pch.anno_simple(df.cluster,add_text=False,legend=True),
    studies=pch.anno_simple(df.studies,add_text=False,legend=False),
    cell_types=pch.anno_simple(df.cell_types, add_text=False,legend=False),axis=0,                       
)

In [ ]:
plt.figure(figsize=(7, 6))
cm = pch.ClusterMapPlotter(
    #data=pd.DataFrame(mat),
    data=pd.DataFrame(mat),
    top_annotation=col_ha,left_annotation=row_ha,
    row_cluster=True,
    col_cluster=True,
    cmap='jet',
    vmin=0,
    vmax=1,
    row_names_side='left',
    #show_rownames=True,
    rasterized=True
    
    
)
plt.savefig("C_clusteringHeatmap", dpi=300, bbox_inches="tight")
plt.show()